# Track 3 Technical Extender

**For:** learners comfortable with Python, tests, and statistical interpretation. **Time:** 4–8 hours.

Validate low-flow methods, enforce completeness, compare estimators, and design provenance. Work here or in a branch; do not overwrite the three track notebooks. **Status: method-development draft.**

## Learning objectives

By the end of the hackathon, students should be able to:

1. **Frame a relevant question** about climate, agriculture or natural-resource stewardship and explain why it interests them.
2. **Use their selected notebook pathway** to explore environmental data, documenting what they tried and any challenges encountered.
3. **Interpret and communicate evidence**, explaining the source, location, time period and meaning of any results or visualizations they present.
4. **Recognize limitations and uncertainty**, distinguishing what the data show from what would require additional evidence.
5. **Propose a future project**, identifying a next question and the data, skills or partnerships needed to pursue it.
6. **Describe potential community benefits** and explain who might find the work useful.
7. **Identify appropriate reviewers or collaborators** and explain how their perspectives could improve interpretation and guide responsible sharing.

These objectives apply across all three tracks. Students demonstrate learning through their final presentations and explanations of completed or attempted work; a finished visualization is not required.

## How we will work

Students work in **one of three tracks**, selected with mentor support: Guided Explorer, Data Investigator or Technical Extender. The tracks are parallel choices, not a sequence to complete. Begin with 15–20 minutes of shared instruction, followed by 5–10 minutes of track-group orientation. Students then start working; mentors provide brief demonstrations when a group needs them. A completed visualization is welcome but is not a condition for presenting or demonstrating learning.

Use the [seven presentation questions](../guides/final_presentation.md) to collect notes as you go.

## Agricultural application

How can regional drought and streamflow records help frame questions about livestock water availability, and what additional local evidence would we need?

Alternative questions may concern grazing, gardens, plant resources or watershed stewardship. State what additional evidence would be needed; regional indicators alone do not establish a management recommendation.

## 1. Test the low-flow definition

A rolling seven-day minimum of daily values and a rolling seven-day mean answer different questions. Test the distinction before applying either method.

In [ ]:
import json
from pathlib import Path
from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
start = Path.cwd().resolve()
REPO = next((p for p in [start, *start.parents] if (p / 'data').is_dir()), None)
DATA = REPO/'data'/'sample_or_fallback' if REPO else None
print(f'Repository: {REPO}')

In [ ]:
def rolling_daily_minimum(values, window=7):
    return pd.Series(values, dtype=float).rolling(window, min_periods=window).min()

def rolling_average(values, window=7):
    return pd.Series(values, dtype=float).rolling(window, min_periods=window).mean()

hand_series = [10, 10, 10, 1, 10, 10, 10]
assert rolling_daily_minimum(hand_series).iloc[-1] == 1
assert np.isclose(rolling_average(hand_series).iloc[-1], 61 / 7)
print('PASS: the two methods differ on the hand-calculated series.')

## 2. Load a gauge and enforce completeness

The ≥330 observed-days rule below is an explicit teaching choice to evaluate.

In [ ]:
SITE_ID = '06446000'
matches = sorted(DATA.glob(f'usgs_nwis_{SITE_ID}_*.rdb')) if DATA else []
if not matches:
    raise FileNotFoundError('Prepared NWIS snapshot missing.')
lines = [line for line in matches[-1].read_text(encoding='utf-8').splitlines() if not line.startswith('#')]
header = lines[0].split('\t')
rows = [line for line in lines[2:] if line.startswith('USGS')]
raw = pd.read_csv(StringIO('\n'.join(rows)), sep='\t', header=None, names=header, dtype=str)
value_column = next(c for c in raw.columns if '00060' in c and not c.endswith('_cd'))
flow = pd.DataFrame({'date': pd.to_datetime(raw['datetime'], errors='coerce'), 'flow_cfs': pd.to_numeric(raw[value_column], errors='coerce')}).dropna().sort_values('date')
flow['year'] = flow['date'].dt.year
flow['flow_7day_mean'] = flow.set_index('date')['flow_cfs'].rolling('7D', min_periods=7).mean().to_numpy()
annual = flow.groupby('year').agg(observed_days=('date','nunique'), annual_7day_low=('flow_7day_mean','min')).reset_index()
annual['usable'] = annual['observed_days'] >= 330
fitted = annual[annual['usable']].dropna(subset=['annual_7day_low'])
assert (fitted['observed_days'] >= 330).all()
print(f'PASS {len(fitted)} usable years; {len(annual)-len(fitted)} excluded years.')
annual.tail()

## 3. Compare estimators

The implementation and narrative must name the same method. Neither fitted line establishes causality.

In [ ]:
x = fitted['year'].to_numpy(dtype=float)
y = fitted['annual_7day_low'].to_numpy(dtype=float)
ols = stats.linregress(x, y)
theil = stats.theilslopes(y, x, alpha=0.95)
comparison = pd.DataFrame([
 {'method':'Ordinary least squares','slope_cfs_decade':ols.slope*10,'lower_95':np.nan,'upper_95':np.nan,'p_value':ols.pvalue},
 {'method':'Theil–Sen','slope_cfs_decade':theil.slope*10,'lower_95':theil.low_slope*10,'upper_95':theil.high_slope*10,'p_value':np.nan}
])
display(comparison)
fig, ax = plt.subplots(figsize=(10,5))
ax.scatter(x,y,label='Annual 7-day low')
ax.plot(x,ols.intercept+ols.slope*x,label='OLS')
ax.plot(x,theil.intercept+theil.slope*x,'--',label='Theil–Sen')
ax.set(xlabel='Year',ylabel='Annual minimum 7-day mean flow (cfs)',title=f'Method comparison USGS {SITE_ID}')
ax.legend(); plt.show()

## 4. Include data provenance before export to align with data standard IEEE 2890-2025 Recommended Practice for Provenance of Indigenous Peoples' Data

Complete all placeholders before saving a figure. Never include sensitive knowledge or credentials.

In [ ]:
provenance = {
 'status':'method-development draft: do not distribute',
 'source':{'steward':'USGS','site_id':SITE_ID,'snapshot_file':matches[-1].name},
 'measure':'annual minimum of complete rolling 7-day mean daily discharge',
 'units':'cubic feet per second',
 'completeness_rule':'≥330 observed days/calendar year; teaching choice requiring review',
 'estimators':['ordinary least squares','Theil–Sen'],
 'limitations':['ADD qualification-code review','ADD local hydrologic interpretation','ADD governance review'],
 'intended_audience':'classroom only'
}
required={'status','source','measure','units','completeness_rule','estimators','limitations','intended_audience'}
assert required <= provenance.keys()
print(json.dumps(provenance, indent=2))

## What to bring to the final presentation

Explain what you tried and learned; show any visualizations and describe their source and meaning. Record uncertainty and future project ideas. A completed figure is not required. Use the seven questions below to prepare.

## Final presentation and stewardship

1. **What question did you explore, and why did it interest you?**
2. **What did you learn?** Describe something about the topic, data or method.
3. **What did you create?** Show any visualizations and explain their source, place, period and meaning. If you did not finish a visualization, explain what you tried and what happened.
4. **What remains uncertain?** Identify a limitation, challenge or unanswered question.
5. **What would you investigate next?** Suggest a future project and the data, skills or partnerships it would need.
6. **Who might benefit from this work?** Explain the potential benefit without claiming an outcome the project has not demonstrated.
7. **Who should review or help interpret this work?** Identify relevant people or roles and why their perspective matters; naming a reviewer does not imply approval.

See the [presentation guidance](../guides/final_presentation.md). Save full stewardship details in the agreed class location. The final hour, September 16 10:45–11:45, is for student sharing and questions.